# E-commerce Customer Behavior Big Data Workflow

This notebook is used as a **workflow documentation notebook**.  
The commands and scripts shown here were executed in **PuTTY / Hadoop Sandbox**, not directly inside Jupyter Notebook.

The purpose of this notebook is to clearly record the complete project workflow:

**HDFS → Apache Pig → Apache Spark MLlib → HBase → Jupyter Visualization**

> Note: Code cells in this notebook are mainly for documentation. They should be copied and executed in PuTTY or the correct Hadoop environment when reproducing the project.

## 1. Project Overview

This project analyzes customer behavior in an e-commerce platform using the Taobao User Behavior Dataset.  
The project applies big data tools to clean raw behavior records, generate user-level features, train purchase prediction models, store processed data in HBase, and create visualizations for report analysis.

The final workflow is:

1. Confirm raw dataset in HDFS.
2. Clean raw data using Apache Pig.
3. Generate user-level features using Apache Spark.
4. Train machine learning models using Spark MLlib.
5. Export user features to TSV format.
6. Import user features into HBase.
7. Use Jupyter Notebook to generate visualizations for the report.

## 2. Dataset Information

The dataset used in this project is a 200,000-record subset of the Taobao User Behavior Dataset.

### HDFS Dataset Path

```text
/user/maria_dev/final_report/UserBehavior_200k.csv
```

### Dataset Columns

| Column | Description |
|---|---|
| `user_id` | Unique user identifier |
| `item_id` | Unique product identifier |
| `category_id` | Product category identifier |
| `behavior_type` | User behavior type |
| `timestamp` | Unix timestamp of the behavior |

### Behavior Types

| Behavior Type | Meaning |
|---|---|
| `pv` | Page view |
| `fav` | Add to favorites |
| `cart` | Add to cart |
| `buy` | Purchase |

## 3. Step 1: Confirm Dataset in HDFS

Before running any processing script, the dataset was checked in HDFS.  
This step confirms that the file exists and previews the first few records.

In [ ]:
# Run in PuTTY / Hadoop terminal

hdfs dfs -ls /user/maria_dev/final_report/UserBehavior_200k.csv

hdfs dfs -cat /user/maria_dev/final_report/UserBehavior_200k.csv | head -5

## 4. Step 2: Create Project Folders

The project folders were created under the user home directory.  
Each folder stores scripts for a different part of the workflow.

In [ ]:
# Run in PuTTY / Hadoop terminal

cd ~

mkdir -p ecommerce_project/pig
mkdir -p ecommerce_project/spark
mkdir -p ecommerce_project/hbase

cd ecommerce_project

## 5. Step 3: Create HDFS Output Directories

Separate HDFS folders were created for processed data and output results.

In [ ]:
# Run in PuTTY / Hadoop terminal

hdfs dfs -mkdir -p /user/maria_dev/ecommerce/processed
hdfs dfs -mkdir -p /user/maria_dev/ecommerce/output

## 6. Step 4: Create Pig Cleaning Script

Apache Pig was used to clean the raw dataset.

The Pig script was created using `vi`:

```bash
cd ~/ecommerce_project/pig
vi ecommerce_cleaning.pig
```

The script performs the following tasks:

- Load raw CSV data from HDFS.
- Define the schema.
- Remove incomplete records.
- Keep only valid behavior types: `pv`, `fav`, `cart`, and `buy`.
- Store cleaned behavior records.
- Generate behavior summary by behavior type.

In [ ]:
-- File: ~/ecommerce_project/pig/ecommerce_cleaning.pig
-- Run this script using Apache Pig in PuTTY

raw_data = LOAD '/user/maria_dev/final_report/UserBehavior_200k.csv'
USING PigStorage(',')
AS (
    user_id:chararray,
    item_id:chararray,
    category_id:chararray,
    behavior_type:chararray,
    timestamp:chararray
);

valid_data = FILTER raw_data BY
    user_id IS NOT NULL AND
    item_id IS NOT NULL AND
    category_id IS NOT NULL AND
    behavior_type IS NOT NULL AND
    timestamp IS NOT NULL;

clean_data = FILTER valid_data BY
    behavior_type == 'pv' OR
    behavior_type == 'fav' OR
    behavior_type == 'cart' OR
    behavior_type == 'buy';

STORE clean_data INTO '/user/maria_dev/ecommerce/processed/cleaned_behavior'
USING PigStorage(',');

group_behavior = GROUP clean_data BY behavior_type;

behavior_summary = FOREACH group_behavior GENERATE
    group AS behavior_type,
    COUNT(clean_data) AS behavior_count;

STORE behavior_summary INTO '/user/maria_dev/ecommerce/output/behavior_summary'
USING PigStorage(',');

## 7. Step 5: Run Pig Cleaning Script

Before running Pig, old output folders were removed to avoid HDFS path conflict.

In [ ]:
# Run in PuTTY / Hadoop terminal

cd ~/ecommerce_project/pig

hdfs dfs -rm -r -f /user/maria_dev/ecommerce/processed/cleaned_behavior
hdfs dfs -rm -r -f /user/maria_dev/ecommerce/output/behavior_summary

pig ecommerce_cleaning.pig

## 8. Step 6: Check Pig Output

After the Pig job finished, the cleaned behavior data and behavior summary were checked.

In [ ]:
# Run in PuTTY / Hadoop terminal

hdfs dfs -ls /user/maria_dev/ecommerce/processed/cleaned_behavior

hdfs dfs -ls /user/maria_dev/ecommerce/output/behavior_summary

hdfs dfs -cat /user/maria_dev/ecommerce/processed/cleaned_behavior/part* | head -10

hdfs dfs -cat /user/maria_dev/ecommerce/output/behavior_summary/part*

### Pig Output Result

The behavior summary produced by Pig was:

| Behavior Type | Count |
|---|---:|
| `pv` | 179,146 |
| `cart` | 11,023 |
| `fav` | 5,759 |
| `buy` | 4,073 |

This confirms that the cleaned dataset was successfully generated and that page views were the most frequent user behavior.

## 9. Step 7: Create HBase Table

HBase was used to store user-level behavior features generated by Spark.

The table design was:

| Table | Row Key | Column Family | Columns |
|---|---|---|---|
| `user_features` | `user_id` | `behavior` | `pv_count`, `fav_count`, `cart_count`, `buy_count`, `total_actions`, `is_buyer` |

In [ ]:
# Run in PuTTY / Hadoop terminal

hbase shell

In [ ]:
# Run inside HBase shell

create 'user_features', 'behavior'

list

exit

If the table already exists and needs to be recreated, the following HBase commands can be used.

In [ ]:
# Run inside HBase shell only if the table needs to be recreated

disable 'user_features'
drop 'user_features'
create 'user_features', 'behavior'
exit

## 10. Step 8: Create Spark ML Script

Spark was used for feature engineering and machine learning.

The Spark script was created using `vi`:

```bash
cd ~/ecommerce_project/spark
vi ecommerce_spark_ml.py
```

The script performs the following tasks:

- Read cleaned behavior data from HDFS.
- Generate user-level behavior features.
- Create the target label `is_buyer`.
- Export user features for HBase import.
- Train Logistic Regression, Decision Tree, and Random Forest models.
- Evaluate the models using AUC, accuracy, F1-score, and confusion matrix.

In [ ]:
# File: ~/ecommerce_project/spark/ecommerce_spark_ml.py
# Run this script using spark-submit in PuTTY

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, sum as spark_sum
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

spark = SparkSession.builder \
    .appName("Ecommerce Customer Behavior Prediction") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

input_path = "/user/maria_dev/ecommerce/processed/cleaned_behavior"

df = spark.read.csv(input_path, header=False, inferSchema=True)
df = df.toDF("user_id", "item_id", "category_id", "behavior_type", "timestamp")

print("===== Cleaned Dataset Schema =====")
df.printSchema()

print("===== Sample Cleaned Data =====")
df.show(10, truncate=False)

print("===== Total Cleaned Records =====")
print(df.count())

print("===== Behavior Type Distribution =====")
behavior_summary = df.groupBy("behavior_type").count().orderBy("behavior_type")
behavior_summary.show()

user_features = df.groupBy("user_id").agg(
    spark_sum(when(col("behavior_type") == "pv", 1).otherwise(0)).alias("pv_count"),
    spark_sum(when(col("behavior_type") == "fav", 1).otherwise(0)).alias("fav_count"),
    spark_sum(when(col("behavior_type") == "cart", 1).otherwise(0)).alias("cart_count"),
    spark_sum(when(col("behavior_type") == "buy", 1).otherwise(0)).alias("buy_count")
)

user_features = user_features.withColumn(
    "total_actions",
    col("pv_count") + col("fav_count") + col("cart_count") + col("buy_count")
)

user_features = user_features.withColumn(
    "is_buyer",
    when(col("buy_count") > 0, 1).otherwise(0)
)

print("===== User-level Features =====")
user_features.show(10, truncate=False)

print("===== Buyer and Non-buyer Distribution =====")
user_features.groupBy("is_buyer").count().show()

hbase_export_path = "/user/maria_dev/ecommerce/output/user_features_hbase_tsv"

user_features_for_hbase = user_features.select(
    col("user_id"),
    col("pv_count"),
    col("fav_count"),
    col("cart_count"),
    col("buy_count"),
    col("total_actions"),
    col("is_buyer")
)

user_features_for_hbase.rdd.map(
    lambda row: str(row["user_id"]) + "\t" +
                str(row["pv_count"]) + "\t" +
                str(row["fav_count"]) + "\t" +
                str(row["cart_count"]) + "\t" +
                str(row["buy_count"]) + "\t" +
                str(row["total_actions"]) + "\t" +
                str(row["is_buyer"])
).saveAsTextFile(hbase_export_path)

print("===== User features exported for HBase import =====")
print(hbase_export_path)

feature_columns = ["pv_count", "fav_count", "cart_count"]

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

ml_data = assembler.transform(user_features).select(
    col("features"),
    col("is_buyer").cast("double").alias("label")
)

print("===== ML Dataset Sample =====")
ml_data.show(10, truncate=False)

train_data, test_data = ml_data.randomSplit([0.7, 0.3], seed=42)

print("===== Training Data Count =====")
print(train_data.count())

print("===== Testing Data Count =====")
print(test_data.count())

models = {
    "Logistic Regression": LogisticRegression(featuresCol="features", labelCol="label", maxIter=10),
    "Decision Tree": DecisionTreeClassifier(featuresCol="features", labelCol="label", maxDepth=5),
    "Random Forest": RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=20, maxDepth=5)
}

binary_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

print("===== Machine Learning Model Results =====")

for model_name, model in models.items():
    print("\n===== " + model_name + " =====")

    trained_model = model.fit(train_data)
    predictions = trained_model.transform(test_data)

    predictions.select("features", "label", "prediction", "probability").show(10, truncate=False)

    auc = binary_evaluator.evaluate(predictions)
    accuracy = accuracy_evaluator.evaluate(predictions)
    f1 = f1_evaluator.evaluate(predictions)

    print("AUC: " + str(round(auc, 4)))
    print("Accuracy: " + str(round(accuracy, 4)))
    print("F1-score: " + str(round(f1, 4)))

    print("Confusion Matrix:")
    predictions.groupBy("label", "prediction").count().show()

spark.stop()

## 11. Step 9: Run Spark ML Script

Before running the Spark script, the old TSV output folder was removed to avoid path conflict.

In [ ]:
# Run in PuTTY / Hadoop terminal

cd ~/ecommerce_project/spark

hdfs dfs -rm -r -f /user/maria_dev/ecommerce/output/user_features_hbase_tsv

spark-submit ecommerce_spark_ml.py

If Spark UI port conflict occurs, run Spark with another port.

In [ ]:
# Run in PuTTY / Hadoop terminal if port conflict occurs

spark-submit --conf spark.ui.port=4050 ecommerce_spark_ml.py

### Spark MLlib Model Results

| Model | AUC | Accuracy | F1-score |
|---|---:|---:|---:|
| Random Forest | 0.8969 | 0.9947 | 0.9944 |
| Decision Tree | 0.8890 | 0.9947 | 0.9944 |
| Logistic Regression | 0.8885 | 0.9947 | 0.9944 |

Random Forest achieved the highest AUC score.  
The input features used for model training were `pv_count`, `fav_count`, and `cart_count`.

## 12. Step 10: Check Spark TSV Output

Spark exported user-level features to the following HDFS path:

```text
/user/maria_dev/ecommerce/output/user_features_hbase_tsv
```

In [ ]:
# Run in PuTTY / Hadoop terminal

hdfs dfs -ls /user/maria_dev/ecommerce/output/user_features_hbase_tsv

hdfs dfs -cat /user/maria_dev/ecommerce/output/user_features_hbase_tsv/part* | head -10

## 13. Step 11: Import Spark Output into HBase

The Spark-generated TSV output was imported into HBase using `ImportTsv`.

In [ ]:
# Run in PuTTY / Hadoop terminal

hbase org.apache.hadoop.hbase.mapreduce.ImportTsv \
-Dimporttsv.separator=$'\t' \
-Dimporttsv.columns=HBASE_ROW_KEY,behavior:pv_count,behavior:fav_count,behavior:cart_count,behavior:buy_count,behavior:total_actions,behavior:is_buyer \
user_features \
/user/maria_dev/ecommerce/output/user_features_hbase_tsv

## 14. Step 12: Verify HBase Import

After importing the TSV file, the HBase table was verified using `scan` and `count`.

In [ ]:
# Run in PuTTY / Hadoop terminal

hbase shell

In [ ]:
# Run inside HBase shell

scan 'user_features', {LIMIT => 10}

count 'user_features'

exit

### HBase Verification Result

The final HBase table contained:

| Item | Result |
|---|---:|
| HBase table | `user_features` |
| Column family | `behavior` |
| Stored user-level records | 170,236 |
| Bad lines during ImportTsv | 0 |

This confirms that Spark-generated user features were successfully imported into HBase.

## 15. Jupyter Visualization Files

The visualizations used in the report were generated in Jupyter Notebook after downloading the cleaned behavior file from HDFS.

The cleaned data file was saved locally as:

```text
cleaned_behavior.csv
```

The generated visualization files were saved in the `images/` folder:

| File | Description |
|---|---|
| `behavior_distribution.png` | User behavior type distribution |
| `conversion_funnel.png` | E-commerce conversion funnel |
| `hourly_user_activity.png` | Hourly user activity pattern |
| `buyer_nonbuyer_distribution.png` | Buyer vs non-buyer distribution |
| `top10_purchase_categories.png` | Top 10 product categories by purchases |
| `model_performance_comparison.png` | Machine learning model performance comparison |
| `confusion_matrix_summary.png` | Confusion matrix summary |

## 16. Final Results Summary

| Component | Result |
|---|---:|
| Cleaned behavior records | 200,001 |
| Unique user-level records | 170,236 |
| Page views | 179,146 |
| Cart actions | 11,023 |
| Favorite actions | 5,759 |
| Purchases | 4,073 |
| Buyers | 4,056 |
| Non-buyers | 166,180 |
| Best model | Random Forest |
| Best AUC | 0.8969 |
| HBase stored records | 170,236 |

## 17. Notes for Reproduction

This notebook is not intended to be executed from top to bottom inside Jupyter Notebook.  
It is a structured record of the commands and scripts used in the project.

To reproduce the project:

1. Run the HDFS, Pig, Spark, and HBase commands in PuTTY / Hadoop Sandbox.
2. Save the Pig script as `pig/ecommerce_cleaning.pig`.
3. Save the Spark script as `spark/ecommerce_spark_ml.py`.
4. Run Jupyter Notebook only for visualization after downloading the cleaned data.
5. Use the generated images in the final report and GitHub README.